# QM when no upper bound is known

In [1]:
import numpy as np
from sim import simulate_errors

from grid_search import evaluate_algorithm, evaluate_algorithm_single, grid_search_algorithm, grid_search_algorithm_single
from brute_force import find_phi_brute_force, find_phi_fixed_budget_brute_force
from binary_search_new import  find_phi_fixed_budget_binary_search, find_phi_fixed_budget_binary_search_delayed

rng = np.random.default_rng(42)
R = 1000

In [2]:
phi_max = np.pi/16
print(f"{phi_max = }")
phi_min = 0.01

N_min = max(np.pi//(2*phi_max), 1)
N_max = max(np.pi//(2*phi_min), 1)

N_min, N_max

phi_max = 0.19634954084936207


(8.0, 157.0)

In [3]:
phi = 0.05
phi_max = 0.1
phi_min = 0.01

def find_phi_fixed_budget_binary_search_delayed(rng, phi, phi_max, phi_min, m_exploration, budget, lookback_window=5, safeguard=1):
    N_min = max(int(np.pi // (2 * phi_max)), 1)
    N_max = max(int(np.pi // (2 * phi_min)), 1)
    N = max(1, N_min)

    phi_hat_list = []
    N_history = []
    budget_used = 0

    def probe(N):
        nonlocal budget_used
        phi_hat = simulate_errors(rng, phi, m_exploration, N)
        budget_used += N * m_exploration
        phi_hat_list.append(phi_hat)
        N_history.append(N)
        return phi_hat

    def overshot():
        if len(phi_hat_list) < lookback_window + 1:
            return False
        means = [np.mean(phi_hat_list[:k+1]) for k in range(len(phi_hat_list))]
        recent = means[-(lookback_window + 1):]
        return all(recent[i] < recent[i-1] for i in range(1, len(recent)))

    lb = N_min
    ub = N_max

    probe(N)
    N += (ub - N) // 2

    done = False
    while not done:
        probe(N)
        temp_N = N

        if overshot():
            # rewind: the first bad N was lookback_window steps ago
            ub = N_history[-lookback_window]
            # last confirmed good N is one step before that
            lb = N_history[-lookback_window - 1] if len(N_history) > lookback_window else N_min
            N = lb + (ub - lb) // 2
        else:
            lb = N
            N += (ub - N) // 2

        if temp_N == N or N < N_min or N > N_max or budget_used >= budget:
            done = True

    remaining_budget = budget - budget_used
    if remaining_budget <= 0:
        return phi_hat_list[-1], budget_used

    N = max(lb - safeguard, 1)

    m = int(remaining_budget / N)
    phi_hat = simulate_errors(rng, phi, m, N)
    budget_used += m * N

    return phi_hat, budget_used


params = {
  "m_exploration": 200,
  "budget": 100_000,
  "lookback_window": 2,
  "safeguard": 5,
}

find_phi_fixed_budget_binary_search_delayed(rng, phi, phi_max, phi_min, **params)

(0.007911371272093792, 107200)

In [4]:
def find_phi_fixed_budget_binary_search_delayed_2(rng, phi, phi_max, phi_min, m_exploration, budget, 
                                                 m_reference=100, lookback_window=5, safeguard=1, overshoot_ratio=0.5):
    N_min = max(int(np.pi // (2 * phi_max)), 1)
    N_max = max(int(np.pi // (2 * phi_min)), 1)

    phi_hat_list = []
    N_history = []
    budget_used = 0

    def probe(N, m):
        nonlocal budget_used
        phi_hat = simulate_errors(rng, phi, m, N)
        budget_used += N * m
        phi_hat_list.append(phi_hat)
        N_history.append(N)
        return phi_hat

    def overshot():
        if len(phi_hat_list) < lookback_window + 1:
            return False
        reference = phi_hat_list[0]                             # single high-m reference at N_min
        recent    = np.nanmean(phi_hat_list[-lookback_window:]) # mean over last low-m probes
        return recent < overshoot_ratio * reference

    lb = N_min
    ub = N_max

    probe(N_min, m_reference)   # high-m anchor
    N = N_min + (ub - N_min) // 2

    done = False
    while not done:
        probe(N, m_exploration)  # cheap search probes
        temp_N = N

        if overshot():
            ub = N_history[-lookback_window]
            lb = N_history[-lookback_window - 1] if len(N_history) > lookback_window else N_min
            N = lb + (ub - lb) // 2
        else:
            lb = N
            N += (ub - N) // 2

        if temp_N == N or N < N_min or N > N_max or budget_used >= budget:
            done = True

    remaining_budget = budget - budget_used
    if remaining_budget <= 0:
        return phi_hat_list[-1], budget_used

    N = max(lb - safeguard, 1)
    m = int(remaining_budget / N)
    phi_hat = simulate_errors(rng, phi, m, N)
    budget_used += m * N

    return phi_hat, budget_used

params = {
  "m_reference": 400,
  "m_exploration": 100,
  "budget": 100_000,
  "lookback_window": 2,
  "safeguard": 2,
}

find_phi_fixed_budget_binary_search_delayed_2(rng, phi, phi_max, phi_min, **params)

(0.01068160230513076, 100900)

In [ ]:
from scipy.stats import norm

from binary_search import find_phi_fixed_budget_binary_search_delayed as find_phi_fixed_budget_binary_search_delayed_3

params = {
  "m_reference": 400,
  "m_exploration": 100,
  "budget": 100_000,
  "lookback_window": 1,
  "safeguard": 2,
}

find_phi_fixed_budget_binary_search_delayed_3(rng, phi, phi_max, phi_min, **params)

(0.04984673289338511, 99991)

In [6]:
rng_test = np.random.default_rng(42)
phi_test = np.random.uniform(phi_min, phi_max)
result = find_phi_fixed_budget_binary_search_delayed_3(
    rng_test, phi_test, phi_max, phi_min,
    m_exploration=100, budget=10000,
    m_reference=400
)
print(result)

(0.009831896565056307, 14600)


In [7]:
phi_max = np.pi/16
print(f"{phi_max = }")
phi_min = 0.01

params = {
  "m_reference": 400,
  "m_exploration": 100,
  "budget": 100_000,
  "lookback_window": 1,
  "safeguard": 1,
}


results = evaluate_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    # find_phi_fn=find_phi_fixed_budget_binary_search,
    params=params,
    R=R,
    phi_max=phi_max,
    phi_min=phi_min,
    eps=1e-3,
    rng=rng,
)

phi_max = 0.19634954084936207


Evaluating: 100%|██████████| 1000/1000 [00:00<00:00, 1397.46it/s]

Total runtime          : 0.01 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.724 (72.4%)
Mean absolute error    : 1.328e-03
Median absolute error  : 5.165e-04
Mean budget            : 100,240
Median budget          : 99,994
------------------------------------------------------------


In [12]:
np.logspace(2, 4, num=25, dtype=int)

array([  100,   121,   146,   177,   215,   261,   316,   383,   464,
         562,   681,   825,  1000,  1211,  1467,  1778,  2154,  2610,
        3162,  3831,  4641,  5623,  6812,  8254, 10000])

In [11]:
np.logspace(1, 2, num=10, dtype=int)

array([ 10,  12,  16,  21,  27,  35,  46,  59,  77, 100])

In [ ]:
param_grid = {
    "m_reference": np.logspace(2, 4, num=10, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6400/6400 [42:31<00:00,  2.51it/s, loss=9.90e-02, success=0.00]  


=== Grid search summary ===
Total runtime          : 42.53 minutes
Total configurations   : 6400
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 1000, 'm_exploration': 16, 'lookback_window': 1, 'safeguard': 2, 'budget': 100000, 'conf': 0.5, 'mean_budget': 99993.328, 'err': 0.000459038854356684, 'success_rate': 0.905, 'loss': 0.000459038854356684}

No configuration met success threshold.


In [13]:
param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [35:56<00:00,  2.78it/s, loss=9.74e-02, success=0.03] 


=== Grid search summary ===
Total runtime          : 35.95 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 2154, 'm_exploration': 12, 'lookback_window': 1, 'safeguard': 1, 'budget': 100000, 'conf': 0.5, 'mean_budget': 99991.779, 'err': 0.0004257004778746689, 'success_rate': 0.939, 'loss': 0.0004257004778746689}

No configuration met success threshold.


In [14]:
phi_max = 0.1
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [33:22<00:00,  3.00it/s, loss=4.44e-02, success=0.10]


✓ Updated best_overall.json
=== Grid search summary ===
Total runtime          : 33.37 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 857

Best by loss:
{'m_reference': 681, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 100000, 'conf': 0.4, 'mean_budget': 99986.738, 'err': 0.0003135121579751796, 'success_rate': 0.975, 'loss': 0.0003135121579751796}

Best by budget (success-constrained):
{'m_reference': 1467, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 100000, 'conf': 0.4, 'mean_budget': 99982.075, 'err': 0.0003268602693560823, 'success_rate': 0.966, 'loss': 0.0003268602693560823}


In [15]:
phi_max = 0.1
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [10_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [20:51<00:00,  4.79it/s, loss=4.61e-02, success=0.09] 

=== Grid search summary ===
Total runtime          : 20.86 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 121, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 10000, 'conf': 0.5, 'mean_budget': 10018.781, 'err': 0.001334167539665838, 'success_rate': 0.491, 'loss': 0.001334167539665838}

No configuration met success threshold.


In [16]:
phi_max = 0.1
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [10_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-4,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [20:20<00:00,  4.91it/s, loss=4.40e-02, success=0.01]


=== Grid search summary ===
Total runtime          : 20.35 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 121, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 10000, 'conf': 0.4, 'mean_budget': 10013.093, 'err': 0.001340614874195919, 'success_rate': 0.043, 'loss': 0.001340614874195919}

No configuration met success threshold.


In [18]:
phi_max = 0.01
phi_min = 0.001

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [10_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-4,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [21:14<00:00,  4.71it/s, loss=4.43e-03, success=0.10]


=== Grid search summary ===
Total runtime          : 21.25 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 146, 'm_exploration': 100, 'lookback_window': 1, 'safeguard': 2, 'budget': 10000, 'conf': 0.4, 'mean_budget': 109222.0, 'err': 0.004155104368768609, 'success_rate': 0.115, 'loss': 0.004155104368768609}

No configuration met success threshold.


In [17]:
phi_max = 0.1
phi_min = 0.001

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [10_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-4,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [19:33<00:00,  5.11it/s, loss=5.03e-02, success=0.01]

=== Grid search summary ===
Total runtime          : 19.55 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 100, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 10000, 'conf': 0.5, 'mean_budget': 13831.22, 'err': 0.046139974709686386, 'success_rate': 0.007, 'loss': 0.046139974709686386}

No configuration met success threshold.


In [20]:
phi_max = np.pi/2
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [10_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [26:35<00:00,  3.76it/s, loss=7.70e-01, success=0.01]


=== Grid search summary ===
Total runtime          : 26.59 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 681, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 3, 'budget': 10000, 'conf': 0.5, 'mean_budget': 10000.887, 'err': 0.004244488624998462, 'success_rate': 0.154, 'loss': 0.004244488624998462}

No configuration met success threshold.


In [19]:
phi_max = np.pi/2
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [51:46<00:00,  1.93it/s, loss=7.90e-01, success=0.00]  

=== Grid search summary ===
Total runtime          : 51.78 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 4641, 'm_exploration': 10, 'lookback_window': 1, 'safeguard': 1, 'budget': 100000, 'conf': 0.6, 'mean_budget': 99998.581, 'err': 0.0011612412470443367, 'success_rate': 0.547, 'loss': 0.0011612412470443367}

No configuration met success threshold.


In [21]:
phi_max = np.pi/4
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)

Grid search (single-threaded): 100%|██████████| 6000/6000 [50:58<00:00,  1.96it/s, loss=4.04e-01, success=0.00]  


=== Grid search summary ===
Total runtime          : 50.97 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 5623, 'm_exploration': 12, 'lookback_window': 1, 'safeguard': 1, 'budget': 100000, 'conf': 0.5, 'mean_budget': 99998.156, 'err': 0.0009965262806510386, 'success_rate': 0.592, 'loss': 0.0009965262806510386}

No configuration met success threshold.


In [22]:
phi_max = np.pi/8
phi_min = 0.01

param_grid = {
    "m_reference": np.logspace(2, 4, num=25, dtype=int),
    "m_exploration": np.logspace(1, 2, num=10, dtype=int),
    "lookback_window": [1, 2],
    "safeguard": [1, 2, 3, 5],
    "budget": [100_000],
    "conf": [0.4, 0.5, 0.6]
}

results = grid_search_algorithm_single(
    find_phi_fn=find_phi_fixed_budget_binary_search_delayed_3,
    param_grid=param_grid,
    R=R,
    phi_min=phi_min,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
    lambda_budget=0,
)


Grid search (single-threaded): 100%|██████████| 6000/6000 [43:23<00:00,  2.30it/s, loss=1.93e-01, success=0.01]  


=== Grid search summary ===
Total runtime          : 43.40 minutes
Total configurations   : 6000
Candidates ≥ 0.95: 0

Best by loss:
{'m_reference': 3162, 'm_exploration': 21, 'lookback_window': 1, 'safeguard': 1, 'budget': 100000, 'conf': 0.4, 'mean_budget': 99995.852, 'err': 0.0006091611249670847, 'success_rate': 0.819, 'loss': 0.0006091611249670847}

No configuration met success threshold.
